In [20]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [22]:
hf_token = os.getenv("HF_TOKEN")   

In [12]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of India?"
        }
    ],
)

print(response.choices[0].message.content)

The capital of India is **New Delhi**.


In [23]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    huggingfacehub_api_token=hf_token,
    provider="auto",
    max_new_tokens=180,
    temperature=0.2,
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [14]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [15]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [16]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [17]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [24]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company needed someone who could *crust* the competition and really *deliver* on the "slice" of the market! 🍕😄',
 'explanation': '**Breaking Down the Joke**\n\n> **“Why did the pizza apply for a job?  \n> Because it heard the company needed someone who could *crust* the competition and really *deliver* on the ‘slice’ of the market! 🍕😄”**\n\nThe humor comes from a blend of two classic joke ingredients:\n\n1. **A playful “set‑up” that invites a punchline** – The first line (“Why did the pizza apply for a job?”) sounds like the kind of absurd “why did the chicken cross the road?” question that primes the listener for a funny, unexpected answer.\n\n2. **A string of puns that mash together pizza‑related vocabulary with business‑speak** – Each highlighted word (in *italics*) has a double meaning that works both in the culinary world of pizza and in the corporate lingo of competition, delivery, and marke

In [25]:
workflow.get_state(config1)


StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company needed someone who could *crust* the competition and really *deliver* on the "slice" of the market! 🍕😄', 'explanation': '**Breaking Down the Joke**\n\n> **“Why did the pizza apply for a job?  \n> Because it heard the company needed someone who could *crust* the competition and really *deliver* on the ‘slice’ of the market! 🍕😄”**\n\nThe humor comes from a blend of two classic joke ingredients:\n\n1. **A playful “set‑up” that invites a punchline** – The first line (“Why did the pizza apply for a job?”) sounds like the kind of absurd “why did the chicken cross the road?” question that primes the listener for a funny, unexpected answer.\n\n2. **A string of puns that mash together pizza‑related vocabulary with business‑speak** – Each highlighted word (in *italics*) has a double meaning that works both in the culinary world of pizza and in the corporate lingo of competition, 

In [27]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company needed someone who could *crust* the competition and really *deliver* on the "slice" of the market! 🍕😄', 'explanation': '**Breaking Down the Joke**\n\n> **“Why did the pizza apply for a job?  \n> Because it heard the company needed someone who could *crust* the competition and really *deliver* on the ‘slice’ of the market! 🍕😄”**\n\nThe humor comes from a blend of two classic joke ingredients:\n\n1. **A playful “set‑up” that invites a punchline** – The first line (“Why did the pizza apply for a job?”) sounds like the kind of absurd “why did the chicken cross the road?” question that primes the listener for a funny, unexpected answer.\n\n2. **A string of puns that mash together pizza‑related vocabulary with business‑speak** – Each highlighted word (in *italics*) has a double meaning that works both in the culinary world of pizza and in the corporate lingo of competition,

## Time Travel

In [30]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b66b4-adcb-6b41-8000-412e088bea82"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b66b4-adcb-6b41-8000-412e088bea82'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-22T09:52:15.893177+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b66b4-adbf-686b-bfff-101d0394ba3b'}}, tasks=(PregelTask(id='3e782ddb-9e56-13f8-b3c1-972e29ed18bc', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error="BadRequestError('(Request ID: Root=1-6ab24f8a-44b236ba16e26b27253328d2;309b9db9-b710-4064-9b66-092f462f90e6)\\n\\nBad request:\\nModel not supported by provider hf-inference')", interrupts=(), state=None, result=None),), interrupts=())

In [31]:
workflow.invoke(None, config={"configurable": {"thread_id": "1", "checkpoint_id": "1f1b66b4-adcb-6b41-8000-412e088bea82"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the position was "a slice of the action" and it wanted to be *topping* the charts! 🍕😄',
 'explanation': '**Why the joke works – a step‑by‑step breakdown**\n\n| Part of the joke | What it says | What it *really* plays on |\n|------------------|--------------|---------------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “Why did …?” riddle format. The listener expects a funny, usually punny reason. | The absurd image of a **pizza** (an inanimate food) behaving like a person looking for work. This incongruity primes the audience for wordplay. |\n| **“Because it heard the position was ‘a slice of the action’ …”** | Explains the pizza’s motivation: it heard the job was a “slice of the action.” | *Slice* is a pizza‑related term (a piece of pizza). “A slice of the action” is an idiom meaning “a share of the excitement or profit.” The joke swaps the idiom’s ordinary meaning for a 

In [32]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the position was "a slice of the action" and it wanted to be *topping* the charts! 🍕😄', 'explanation': '**Why the joke works – a step‑by‑step breakdown**\n\n| Part of the joke | What it says | What it *really* plays on |\n|------------------|--------------|---------------------------|\n| **“Why did the pizza apply for a job?”** | Sets up a classic “Why did …?” riddle format. The listener expects a funny, usually punny reason. | The absurd image of a **pizza** (an inanimate food) behaving like a person looking for work. This incongruity primes the audience for wordplay. |\n| **“Because it heard the position was ‘a slice of the action’ …”** | Explains the pizza’s motivation: it heard the job was a “slice of the action.” | *Slice* is a pizza‑related term (a piece of pizza). “A slice of the action” is an idiom meaning “a share of the excitement or profit.” The joke swaps the idiom’s ord

## Updating state

In [34]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b66b4-adcb-6b41-8000-412e088bea82", "checkpoint_ns":""}}, {"topic":"burger"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b6714-ece6-6a5d-8001-8f84a377acd1'}}

In [35]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b6714-ece6-6a5d-8001-8f84a377acd1'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-22T10:35:19.490620+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b66b4-adcb-6b41-8000-412e088bea82'}}, tasks=(PregelTask(id='4b9e26d8-4e52-8a98-22ca-7480886b268a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the position was "a slice of the action" and it wanted to be *topping* the charts! 🍕😄', 'explanation': '**Why the joke works – a step‑by‑step breakdown**\n\n| Part of the joke | What it says | What it *really* plays on |\n|------------------|--------------|------------------

In [36]:
workflow.invoke(None, config={"configurable": {"thread_id": "1", "checkpoint_id": "1f1b6714-ece6-6a5d-8001-8f84a377acd1"}})

{'topic': 'burger',
 'joke': "Why did the burger go to therapy?  \n\nBecause it couldn't stop **ketch**ing up with its feelings and was always feeling a little *bun*der the weather!",
 'explanation': "**The joke broken down**\n\n> *Why did the burger go to therapy?*  \n> *Because it couldn't stop **ketch**ing up with its feelings and was always feeling a little *bun*der the weather!*\n\n---\n\n### 1. The set‑up: “Why did the burger go to therapy?”\nThe first line sets up a classic “why did X do Y?” format.  \nIn jokes of this type, the answer usually contains a pun or a play on words that links the subject (a burger) to a human situation (needing therapy).\n\n### 2. The punchline: two layered puns\nThe answer is actually made of **two separate wordplays**, each anchored in burger‑related vocabulary.\n\n| Phrase in the punchline | What it sounds like / literal meaning | How it becomes a joke |\n|--------------------------|----------------------------------------|-----------------------|